# AgentCore Memory Tool을 사용하는 Strands Multi-Agent System(단기 메모리) - MemoryManager 사용

## 소개

이 Notebook에서는 AWS AgentCore Memory와 Strands framework를 사용하여 **공유 메모리를 갖춘 Multi-Agent System**을 구현하는 방법을 살펴봅니다. 앞선 예제에서는 Single-Agent 메모리에 중점을 두었지만, 여기서는 여러 전문 Agent가 공통 Memory 저장소에 액세스하면서 협업하는 방법을 다룹니다.

**참고: 이 예제는 기존 MemoryClient 대신 MemoryManager와 MemorySessionManager를 사용하는 단기 메모리 버전입니다.**

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화                                                        |
| Agent 사용 사례       | Travel Planning Assistant                                                        |
| Agentic Framework   | Strands Agents                                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                   |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, Strands Agents, Memory retrieval via Tool           |
| 예제 난이도  | 초급                                                                         |


학습 내용:

- 여러 Agent가 액세스할 수 있는 공유 Memory 리소스를 설정하는 방법
- 자체 메모리 액세스 권한이 있는 전문 Agent를 도구로 생성하는 방법
- 전문 Agent에 작업을 위임하는 Coordinator Agent를 구현하는 방법
- 여러 Agent 상호작용에서 대화 컨텍스트를 유지하는 방법

### 시나리오 배경

이 예제에서는 다음으로 구성된 **여행 계획 시스템**을 만듭니다.
1. 항공 여행을 전문으로 하는 Flight Booking Assistant
2. 숙박을 담당하는 Hotel Booking Assistant
3. 전문 Agent에 작업을 위임하는 Travel Coordinator

이 접근 방식은 복잡한 도메인을 동일한 Memory 저장소를 공유하는 전문 Agent로 분할하는 방법을 보여 줍니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항
- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하고 공유 Memory 리소스를 생성하겠습니다.

## 1단계: 환경 설정
Notebook 실행에 필요한 모든 라이브러리를 가져오고 client를 정의합니다.

In [1]:
!pip install -qr requirements.txt

In [2]:
import logging
import os
from datetime import datetime
from strands.hooks import (
    AgentInitializedEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

# 메모리 관리 모듈 가져오기
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

Amazon Bedrock 모델 및 AgentCore에 적절한 권한이 있는 리전과 역할을 정의합니다.

In [3]:
REGION = os.getenv("AWS_REGION", "us-west-2")
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("agentcore-memory")

## 2단계: 공유 Memory 생성
이 섹션에서는 전문 Agent가 공유할 Memory 리소스를 생성합니다.

In [ ]:
memory_manager = MemoryManager(region_name=REGION)

try:
    print("Creating Memory...")
    memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")

    # Memory 리소스 생성
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for travel agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간
        memory_execution_role_arn=None,  # 단기 메모리에서는 선택 사항
    )

    # Memory ID 추출 및 출력
    memory_id = memory.id
    logger.info("✅ Successfully created/retrieved memory with MemoryManager:")
    logger.info(f"   Memory ID: {memory_id}")
    logger.info(f"   Memory Name: {memory.name}")
    logger.info(f"   Memory Status: {memory.status}")
except Exception as e:
    # 향상된 오류 보고와 함께 Memory 생성 오류 처리
    logger.error(f"❌ Memory creation failed: {e}")
    logger.error(f"Error type: {type(e).__name__}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if "memory_id" in locals():
        try:
            logger.info(f"Attempting cleanup of partially created memory: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"✅ Successfully cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"❌ Failed to clean up memory: {cleanup_error}")

    # 원래 예외 다시 발생
    raise

### Multi-Agent System의 공유 Memory 이해

생성한 Memory 리소스는 여행 계획 시스템의 공유 지식 기반 역할을 합니다. 모든 Agent가 이 공통 Memory 저장소를 읽고 쓰므로 다음이 가능해집니다.

1. **지식 일관성**: 모든 Agent가 동일한 정보로 작업
2. **컨텍스트 보존**: Agent 전환 중에도 대화 기록 유지
3. **전문 액세스**: 각 Agent는 자체 actor_id를 사용하지만 session_id는 공유

이 접근 방식을 사용하면 전문 Agent가 전체 대화 컨텍스트를 활용하면서도 담당 도메인에 집중할 수 있습니다.

## 3단계: Session Manager 초기화

이 섹션에서는 세션 기반 메모리 작업을 위한 MemorySessionManager를 소개하고 actor와 session을 관리할 MemorySession을 생성합니다.

In [ ]:
# Session Memory Manager 초기화
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

logger.info(f"✅ Session manager initialized for memory: {memory.id}")
logger.info(f"Session manager type: {type(session_manager)}")

## 4단계: Memory Hook Provider 생성

이 단계에서는 메모리 작업을 자동화하는 사용자 지정 `MemoryHookProvider` 클래스를 정의합니다. Hook은 Agent 실행 수명 주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 만드는 Memory Hook에는 두 가지 주요 기능이 있습니다.

1. **Memory 검색**: 사용자가 메시지를 보내면 관련된 과거 대화를 자동으로 가져옴
2. **Memory 저장**: Agent가 응답한 후 새 대화를 저장

**MemoryClient 버전과 비교한 주요 변경 사항:**
- MemoryClient 대신 MemorySession 사용
- tuple 대신 ConversationalMessage 객체 사용
- create_event() 대신 add_turns() 사용
- type safety를 위해 MessageRole enum 사용

In [6]:
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_session: MemorySession, memory_id: str):
        self.memory_session = memory_session
        self.memory_id = memory_id

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        try:
            # 사전 구성된 Memory Session 사용(actor_id/session_id 불필요)
            recent_turns = self.memory_session.get_last_k_turns(k=5)

            if recent_turns:
                # 대화 기록을 컨텍스트 형식으로 변환
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # EventMessage 객체와 dict 형식을 모두 처리
                        if hasattr(message, "role") and hasattr(message, "content"):
                            role = message["role"]
                            content = message["content"]
                        else:
                            role = message.get("role", "unknown")
                            content = message.get("content", {}).get("text", "")
                        context_messages.append(f"{role}: {content}")

                context = "\n".join(context_messages)
                logger.info(f"Context from memory: {context}")

                # Agent의 system prompt에 컨텍스트 추가
                event.agent.system_prompt += f"\n\nRecent conversation history:\n{context}\n\nContinue the conversation naturally based on this context."
                logger.info(f"✅ Loaded {len(recent_turns)} recent conversation turns")
            else:
                logger.info("No previous conversation history found")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """MemorySession으로 메시지를 메모리에 저장합니다."""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT

                # Memory Session 인스턴스 사용(actor_id/session_id 전달 불필요)
                result = self.memory_session.add_turns(messages=[ConversationalMessage(message_text, message_role)])

                event_id = result["eventId"]
                logger.info(f"✅ Stored message with Event ID: {event_id}, Role: {message_role.value}")

        except Exception as e:
            logger.error(f"Memory save error: {e}")
            import traceback

            logger.error(f"Full traceback: {traceback.format_exc()}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Memory Hook 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 5단계: Strands Agents로 Multi-Agent 아키텍처 생성
이 섹션에서는 항공편 및 호텔 예약 전문 Agent로 Multi-Agent System을 생성합니다. 두 Agent는 모두 Memory 리소스에 대한 액세스를 공유합니다.

In [7]:
# 필요한 구성 요소 가져오기
from strands import Agent, tool

In [8]:
# 각 전문 Agent에 고유 Actor ID를 생성하되 Session ID는 공유
FLIGHT_ACTOR_ID = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
HOTEL_ACTOR_ID = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

### Memory 액세스가 있는 전문 Agent 생성

다음으로 전문 Agent의 system prompt를 정의합니다. 각 prompt에는 Agent가 파싱할 수 있는 형식으로 Memory parameter가 포함됩니다.

In [9]:
# Hotel Booking 전문 Agent용 system prompt
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# Flight Booking 전문 Agent용 system prompt
FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

### Agent Tool 구현
이제 Coordinator Agent가 사용할 수 있도록 전문 Agent를 도구로 구현합니다.

In [10]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        # Booking Assistant용 Memory Session 생성
        memory_session = session_manager.create_memory_session(actor_id=FLIGHT_ACTOR_ID, session_id=SESSION_ID)
        flight_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": FLIGHT_ACTOR_ID, "session_id": SESSION_ID},
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"


@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        # Booking Assistant용 Memory Session 생성
        memory_session = session_manager.create_memory_session(actor_id=HOTEL_ACTOR_ID, session_id=SESSION_ID)

        hotel_memory_hooks = ShortTermMemoryHook(memory_session, memory_id)

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": HOTEL_ACTOR_ID, "session_id": SESSION_ID},
        )

        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Coordinator Agent 생성

마지막으로 이러한 전문 도구를 조율하는 기본 Travel Planning Agent를 생성합니다.

In [11]:
# Coordinator Agent용 system prompt
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [12]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
)

2025-10-10 23:05:10 - INFO - Found credentials in environment variables.


#### Multi-Agent System이 준비되었습니다!

## Agent 테스트

여행 계획 시나리오로 Multi-Agent System을 테스트해 보겠습니다.

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
response = travel_agent(
    "I would only like to focus on the flight at the moment. direct flimid-range, city center, pool, standard room"
)

## 메모리 지속성 테스트

메모리 시스템이 올바르게 작동하는지 확인하기 위해 새 Travel Agent 인스턴스를 생성하고 이전에 저장한 정보에 액세스할 수 있는지 살펴보겠습니다.

In [ ]:
# 새 Travel Agent 인스턴스 생성
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant],
)

# 이전 대화에 관해 질문
new_travel_agent("Can you remind me about flights talked about before?")

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

1. 여러 Agent가 사용할 공유 Memory 리소스를 생성하는 방법
2. Memory 액세스가 있는 전문 Agent를 도구로 구현하는 방법
3. 대화 컨텍스트를 유지하면서 여러 Agent를 조율하는 방법
4. 서로 다른 Agent 인스턴스 간에 Memory가 유지되는 방식

공유 Memory를 사용하는 이 Multi-Agent 아키텍처는 일관된 사용자 경험을 유지하면서 전문 도메인을 처리할 수 있는 복잡한 대화형 AI 시스템을 구축하는 강력한 접근 방식을 제공합니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# MemoryManager로 Memory 리소스를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"✅ Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")